This notebook contains a script to query the VAST pipeline to get all point like sources within a degree of each pulsar. We use these as candidates to be control sources.

In [1]:
# importing required modules
import matplotlib.pyplot as plt
from vasttools.query import Query
import pandas as pd

from vasttools.moc import VASTMOCS
from astropy import units as u

import numpy as np

from astropy.coordinates import SkyCoord

import scipy.stats as stats

In [2]:
psr_df = pd.read_csv('paper_df_VASTcoverage.csv')

In [ ]:
unfilled_psrs = ['J1752-2806',
 'J1803-2137',
 'J1717-3425',
    ]

psr_df = psr_df[psr_df['JNAME'].isin(unfilled_psrs)]

In [3]:
psr_df

,Unnamed: 0,DOF,p_value,reduced_chi_squared,reduced_chi_squared_err,modulation,modulation_err,chi_squared,chi_squared_err,GB,S1400,DM,P0,P1,AGE,DECJD,ASSOC,JNAME,RAJD
0,176,40,0.000000,2671.566747,16.485002,6.962338,0.076679,106862.669861,659.400073,-0.194956,300.000,478.660,0.455078,2.008920e-14,3.589132e+05,-45.986031,NaN,J1644-4559,251.205304
1,235,12,0.000000,4632.351515,41.018698,28.939939,2.325090,55588.218175,492.224381,-0.961032,47.800,50.372,0.562558,8.129130e-15,1.096450e+06,-28.110361,NaN,J1752-2806,268.244540
2,20,41,0.000000,540.357654,7.347649,14.884816,0.456145,22154.663796,301.253620,-1.018678,20.000,180.440,0.106769,1.510247e-14,1.120117e+05,-49.218056,GRS,J0908-4913,137.147750
3,130,40,0.000000,494.386849,7.117671,46.119800,3.369295,19775.473949,284.706823,1.944153,6.800,24.820,1.368882,1.428802e-15,1.517959e+07,-53.572128,NaN,J1534-5334,233.534492
4,228,47,0.000000,370.368514,5.675784,24.347607,0.891826,17407.320158,266.761857,-0.963350,21.000,88.373,0.367434,1.067181e-14,5.455155e+05,-30.673028,NaN,J1745-3040,266.484650
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
126,294,5,0.966433,0.190167,0.421435,7.950592,20.337828,0.950836,2.107176,-4.871415,0.486,114.544,0.005471,6.209000e-21,1.396167e+10,-14.803503,NaN,J1843-1448,280.755729
127,145,2,0.660059,0.415426,1.055227,11.449910,45.571292,0.830851,2.110454,3.371841,0.340,113.803,0.006284,1.900000e-20,5.240469e+09,-49.630484,NaN,J1552-4937,238.055295
128,92,3,0.898704,0.196686,0.566452,8.306860,19.097985,0.590059,1.699355,2.431279,0.550,354.800,1.213381,5.986000e-15,3.211638e+06,-59.416917,NaN,J1355-5925,208.996292
129,123,4,0.968111,0.138320,0.407575,7.085693,28.002911,0.553278,1.630299,-4.895562,0.320,250.000,1.254052,5.960326e-15,3.333584e+06,-63.138750,NaN,J1519-6308,229.789833


In [4]:
cand_ctrls = {}

In [6]:
for name in psr_df['JNAME'].values:
    psr_name = name
    
    query_coord = SkyCoord(str(psr_df[psr_df['JNAME']==psr_name].iloc[0]['RAJD']) + " " + str(psr_df[psr_df['JNAME']==psr_name].iloc[0]['DECJD']), unit='deg')
    query = Query(coords=query_coord, epochs='23', crossmatch_radius=3600., search_around_coordinates=True, use_tiles=True, corrected_data=False)
    query.find_sources()
    
    num_rows_unmasked = query.results.shape[0]
    point_source_mask = query.results['flux_int'] < (1.5 * query.results['flux_peak'])    
    num_rows_masked = query.results[point_source_mask].shape[0]
    
    cand_ctrls[psr_name] = query.results[point_source_mask]
    
    print(name)
    print("        " + str(num_rows_unmasked) + " candidates")
    print("        " + str(num_rows_masked) + " point source candidates")

J1644-4559
        2227 candidates
        1114 point source candidates
J1752-2806
        752 candidates
        532 point source candidates
J0908-4913
        2352 candidates
        1780 point source candidates
J1534-5334
        1039 candidates
        832 point source candidates
J1745-3040
        1635 candidates
        960 point source candidates
J1605-5257
        1418 candidates
        875 point source candidates
J1804-2717
        1967 candidates
        1384 point source candidates
J1604-4909
        2429 candidates
        1877 point source candidates
J1428-5530
        2771 candidates
        2108 point source candidates
J1001-5507
        2695 candidates
        2063 point source candidates
J1722-3207
        2615 candidates
        1857 point source candidates
J1806-1154
        2645 candidates
        2019 point source candidates
J0907-5157
        3135 candidates
        2444 point source candidates
J1717-4054
        1511 candidates
        1224 point source candidat

In [7]:
vals = []
names = list(cand_ctrls.keys())
for name in names:
    vals.append(cand_ctrls[name])
psr_meas_df = pd.concat(vals, keys=names)

In [8]:
psr_meas_df.to_csv('all_cand_ctrls.csv')